# 177. LayerSkip：Early Exit 与 Self-Speculative Decoding 怎样实现？

> **面试问题：怎样让中间层直接预测 token，并用剩余层验证？early-exit loss、layer dropout、接受率和缓存复用如何设计？**

## 先给结论

普通 Transformer 的中间层没有被训练成可靠输出。LayerSkip 用逐层 dropout 和共享 LM head 的 early-exit loss提升浅层可解码性；推理时浅层充当 draft、完整剩余层验证。加速取决于接受率、共享计算和验证组织，不能只用浅层准确率推断。

## 推荐回答主线

1. 构造多层 decoder，所有层共享最终 LM head，显式返回每层 logits。
2. 对不同深度施加有效 token loss，并用随深度变化的 layer dropout 训练旁路鲁棒性。
3. 实现置信 early exit 与浅层 draft→深层 verify，证明最终提交遵守 full model 决策。
4. 评估 exit-depth、接受长度、质量、吞吐、KV/hidden 复用和错误回退，绑定训练 recipe。

## 教学实现边界

Tiny 模型用前缀均值和残差 MLP 代替完整 attention，以隔离逐层输出/验证语义；greedy verifier 不等于完整随机 speculative sampling 的分布校正。

## 一手资料

- [LayerSkip](https://arxiv.org/abs/2404.16710)
- [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [DeeBERT Early Exit](https://arxiv.org/abs/2004.12993)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。

import warnings  # 导入本单元需要的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元需要的依赖。
import torch.nn.functional as F  # 导入本单元需要的依赖。
from torch import nn  # 导入本单元需要的依赖。

# 小词表和 padding mask 用来验证逐层 loss 与 causal 输入合同。
torch.manual_seed(177)  # 计算并保存当前步骤的中间状态。
VOCAB, DIM, LAYERS = 19, 12, 5  # 计算并保存当前步骤的中间状态。
tokens = torch.tensor([[1, 2, 3, 4, 0], [5, 6, 7, 8, 9]])  # 计算并保存当前步骤的中间状态。
valid = tokens.ne(0)  # 计算并保存当前步骤的中间状态。

assert tokens.shape == valid.shape  # 用受控断言验证关键不变量。
assert LAYERS >= 3  # 用受控断言验证关键不变量。
assert valid.sum().item() == 9  # 用受控断言验证关键不变量。


## 1. 共享 LM head 的多出口模型

LayerSkip 不必为每层增加独立词表头；共享 final norm/head 可降低额外参数，并迫使中间表示对同一输出空间可读。这里的前缀均值保证位置 t 不看未来 token。


In [ ]:
class ResidualLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.ffn = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))  # 计算并保存当前步骤的中间状态。

    def forward(self, hidden):  # 定义本节可复用的核心函数。
        return hidden + self.ffn(self.norm(hidden))  # 返回当前分支计算出的结果。

class TinyLayerSkipLM(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab, dim, layers):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)  # 计算并保存当前步骤的中间状态。
        self.layers = nn.ModuleList(ResidualLayer(dim) for _ in range(layers))  # 计算并保存当前步骤的中间状态。
        self.final_norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.lm_head = nn.Linear(dim, vocab, bias=False)  # 计算并保存当前步骤的中间状态。

    def prefix_hidden(self, token_ids, valid_mask):  # 定义本节可复用的核心函数。
        embedding = self.embedding(token_ids) * valid_mask[..., None]  # 计算并保存当前步骤的中间状态。
        count = torch.cumsum(valid_mask, 1).clamp_min(1)[..., None]  # 计算并保存当前步骤的中间状态。
        return torch.cumsum(embedding, 1) / count  # 返回当前分支计算出的结果。

    def run_layers(self, hidden, start=0, stop=None, layer_path=None):  # 定义本节可复用的核心函数。
        stop = len(self.layers) if stop is None else stop  # 计算并保存当前步骤的中间状态。
        if not 0 <= start <= stop <= len(self.layers):  # 按当前条件选择后续控制路径。
            raise ValueError("层区间非法")  # 遇到非法合同立即显式失败。
        if layer_path is None:  # 按当前条件选择后续控制路径。
            layer_path = torch.ones(len(self.layers), dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
        if len(layer_path) != len(self.layers):  # 按当前条件选择后续控制路径。
            raise ValueError("layer_path 长度必须等于层数")  # 遇到非法合同立即显式失败。
        states = []  # 计算并保存当前步骤的中间状态。
        for layer_id in range(start, stop):  # 遍历输入元素以累积或检查结果。
            if bool(layer_path[layer_id]):  # 按当前条件选择后续控制路径。
                hidden = self.layers[layer_id](hidden)  # 计算并保存当前步骤的中间状态。
            states.append(hidden)  # 计算并保存当前步骤的中间状态。
        return hidden, states  # 返回当前分支计算出的结果。

    def logits_from(self, hidden):  # 定义本节可复用的核心函数。
        return self.lm_head(self.final_norm(hidden))  # 返回当前分支计算出的结果。

    def forward(self, token_ids, valid_mask, layer_path=None, return_hidden=False):  # 定义本节可复用的核心函数。
        hidden = self.prefix_hidden(token_ids, valid_mask)  # 计算并保存当前步骤的中间状态。
        _, states = self.run_layers(hidden, layer_path=layer_path)  # 计算并保存当前步骤的中间状态。
        exits = [self.logits_from(state) for state in states]  # 计算并保存当前步骤的中间状态。
        return (exits, states) if return_hidden else exits  # 返回当前分支计算出的结果。

# forward hook 记录每次多出口调用的同一参数地址，证明所有出口真正复用一个输出头。
model = TinyLayerSkipLM(VOCAB, DIM, LAYERS)  # 计算并保存当前步骤的中间状态。
head_weight_addresses = []  # 计算并保存当前步骤的中间状态。
hook = model.lm_head.register_forward_pre_hook(lambda module, args: head_weight_addresses.append(module.weight.data_ptr()))  # 计算并保存当前步骤的中间状态。
exit_logits, exit_hidden = model(tokens, valid, return_hidden=True)  # 计算并保存当前步骤的中间状态。
hook.remove()  # 计算并保存当前步骤的中间状态。
assert len(exit_logits) == LAYERS  # 用受控断言验证关键不变量。
assert all(logits.shape == (*tokens.shape, VOCAB) for logits in exit_logits)  # 用受控断言验证关键不变量。
assert len(head_weight_addresses) == LAYERS and len(set(head_weight_addresses)) == 1  # 用受控断言验证关键不变量。
assert torch.allclose(exit_logits[0], model.logits_from(exit_hidden[0]))  # 用受控断言验证关键不变量。


## 2. Early-exit loss：所有出口预测相同 next-token 目标

目标仍是 causal shift；padding 和最后无目标位置 mask。不同层 loss 可设权重，浅层太强会伤害最终层，太弱则不可用。每层按自身有效 token 数归一化。


In [ ]:
def shifted_targets(token_ids, valid_mask):  # 定义本节可复用的核心函数。
    targets = torch.zeros_like(token_ids)  # 计算并保存当前步骤的中间状态。
    mask = torch.zeros_like(valid_mask)  # 计算并保存当前步骤的中间状态。
    targets[:, :-1] = token_ids[:, 1:]  # 计算并保存当前步骤的中间状态。
    mask[:, :-1] = valid_mask[:, :-1] & valid_mask[:, 1:]  # 计算并保存当前步骤的中间状态。
    return targets, mask  # 返回当前分支计算出的结果。

def multi_exit_loss(logits_by_layer, targets, mask, weights):  # 定义本节可复用的核心函数。
    losses = []  # 计算并保存当前步骤的中间状态。
    for logits in logits_by_layer:  # 遍历输入元素以累积或检查结果。
        token_loss = F.cross_entropy(logits.reshape(-1, VOCAB), targets.reshape(-1), reduction="none")  # 计算并保存当前步骤的中间状态。
        losses.append(token_loss[mask.reshape(-1)].mean())  # 计算并保存当前步骤的中间状态。
    stacked = torch.stack(losses)  # 计算并保存当前步骤的中间状态。
    return (stacked * weights).sum() / weights.sum(), stacked  # 返回当前分支计算出的结果。

# 总 loss 等于显式加权平均，每层都有有限监督。
targets, target_mask = shifted_targets(tokens, valid)  # 计算并保存当前步骤的中间状态。
weights = torch.linspace(0.2, 1.0, LAYERS)  # 计算并保存当前步骤的中间状态。
loss, layer_losses = multi_exit_loss(exit_logits, targets, target_mask, weights)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(loss, (layer_losses * weights).sum() / weights.sum())  # 用受控断言验证关键不变量。
assert torch.isfinite(layer_losses).all()  # 用受控断言验证关键不变量。
assert target_mask.sum().item() == 7  # 用受控断言验证关键不变量。


## 3. Layer dropout：越深层可用更高跳过概率

训练随机跳层让后续层和输出头适应缺失中间计算。dropout 决策应对整个残差层生效，旁路保持 hidden 不变；评测关闭随机性。下面只生成 schedule 与一次可复现路径。


In [ ]:
def layer_drop_schedule(layers, low=0.0, high=0.4):  # 定义本节可复用的核心函数。
    if layers <= 0 or not 0 <= low <= high < 1:  # 按当前条件选择后续控制路径。
        raise ValueError("dropout schedule 参数非法")  # 遇到非法合同立即显式失败。
    return torch.linspace(low, high, layers)  # 返回当前分支计算出的结果。

def sampled_layer_path(probabilities, generator):  # 定义本节可复用的核心函数。
    if probabilities.ndim != 1 or not ((0 <= probabilities) & (probabilities < 1)).all():  # 按当前条件选择后续控制路径。
        raise ValueError("drop probabilities 必须是一维 [0,1) 张量")  # 遇到非法合同立即显式失败。
    path = torch.rand(len(probabilities), generator=generator) >= probabilities  # 计算并保存当前步骤的中间状态。
    path[-1] = True  # 保留最终层，避免整条路径全空。
    return path  # 返回当前分支计算出的结果。

# schedule 与采样可复现；随后用 hook 和梯度证明被 drop 的层没有执行。
drop_prob = layer_drop_schedule(LAYERS)  # 计算并保存当前步骤的中间状态。
path1 = sampled_layer_path(drop_prob, torch.Generator().manual_seed(7))  # 计算并保存当前步骤的中间状态。
path2 = sampled_layer_path(drop_prob, torch.Generator().manual_seed(7))  # 计算并保存当前步骤的中间状态。
assert torch.all(drop_prob[1:] >= drop_prob[:-1])  # 用受控断言验证关键不变量。
assert torch.equal(path1, path2)  # 用受控断言验证关键不变量。
assert path1[-1]  # 用受控断言验证关键不变量。

drop_path = torch.ones(LAYERS, dtype=torch.bool); dropped_layer = 2; drop_path[dropped_layer] = False  # 计算并保存当前步骤的中间状态。
drop_calls = []  # 计算并保存当前步骤的中间状态。
drop_hook = model.layers[dropped_layer].register_forward_hook(lambda module, args, output: drop_calls.append(1))  # 计算并保存当前步骤的中间状态。
drop_logits, drop_hidden = model(tokens, valid, layer_path=drop_path, return_hidden=True)  # 计算并保存当前步骤的中间状态。
drop_hook.remove()  # 计算并保存当前步骤的中间状态。
assert drop_calls == []  # 用受控断言验证关键不变量。
assert torch.equal(drop_hidden[dropped_layer], drop_hidden[dropped_layer - 1])  # 用受控断言验证关键不变量。

active_calls = []  # 计算并保存当前步骤的中间状态。
active_hook = model.layers[dropped_layer].register_forward_hook(lambda module, args, output: active_calls.append(1))  # 计算并保存当前步骤的中间状态。
model(tokens, valid, layer_path=torch.ones(LAYERS, dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
active_hook.remove()  # 计算并保存当前步骤的中间状态。
assert active_calls == [1]  # 用受控断言验证关键不变量。

model.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
drop_logits[-1][target_mask].square().mean().backward()  # 计算并保存当前步骤的中间状态。
assert model.layers[dropped_layer].ffn[0].weight.grad is None  # 用受控断言验证关键不变量。
assert model.layers[-1].ffn[0].weight.grad is not None and model.layers[-1].ffn[0].weight.grad.norm() > 0  # 用受控断言验证关键不变量。


## 4. 梯度合同：浅层 loss 与最终层 loss 都更新共享表示

多出口监督应让 embedding、早期层与共享 head 收到梯度。监控还要比较各层梯度夹角和最终层回归，避免浅层目标支配训练。


In [ ]:
# 反传加权多出口 loss，检查共享参数与首末层梯度。
model.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
loss.backward()  # 计算并保存当前步骤的中间状态。
assert model.embedding.weight.grad is not None and model.embedding.weight.grad.norm() > 0  # 用受控断言验证关键不变量。
assert model.layers[0].ffn[0].weight.grad is not None  # 用受控断言验证关键不变量。
assert model.layers[-1].ffn[0].weight.grad is not None  # 用受控断言验证关键不变量。


## 5. 置信 early exit：阈值必须在校准集选择

只做低风险分类/抽取时，可在某层最大概率超过阈值后退出。原始 softmax 常过度自信，阈值要按 slice 校准并设置最大允许深度；生成任务的错误会累积，更适合验证式自推测。


In [ ]:
def choose_exit(logits_by_layer, position, threshold, min_layer=1):  # 定义本节可复用的核心函数。
    batch = logits_by_layer[0].shape[0]  # 计算并保存当前步骤的中间状态。
    chosen_layer = torch.full((batch,), len(logits_by_layer), dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    final_probability = torch.softmax(logits_by_layer[-1][:, position], -1)  # 计算并保存当前步骤的中间状态。
    chosen_token = final_probability.argmax(-1)  # 计算并保存当前步骤的中间状态。
    unresolved = torch.ones(batch, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for layer_id, logits in enumerate(logits_by_layer, 1):  # 遍历输入元素以累积或检查结果。
        probability = torch.softmax(logits[:, position], -1)  # 计算并保存当前步骤的中间状态。
        confident = probability.max(-1).values >= threshold  # 计算并保存当前步骤的中间状态。
        take = unresolved & confident & (layer_id >= min_layer)  # 计算并保存当前步骤的中间状态。
        chosen_layer[take] = layer_id  # 计算并保存当前步骤的中间状态。
        chosen_token[take] = probability.argmax(-1)[take]  # 计算并保存当前步骤的中间状态。
        unresolved = unresolved & ~take  # 计算并保存当前步骤的中间状态。
    return chosen_layer, chosen_token  # 返回当前分支计算出的结果。

# 每个样本独立退出；低阈值在最早允许层退出，高阈值逐样本回退最终层。
early_layer, early_token = choose_exit(exit_logits, 2, threshold=0.0, min_layer=2)  # 计算并保存当前步骤的中间状态。
late_layer, late_token = choose_exit(exit_logits, 2, threshold=1.1, min_layer=2)  # 计算并保存当前步骤的中间状态。
assert early_layer.tolist() == [2, 2]  # 用受控断言验证关键不变量。
assert late_layer.tolist() == [LAYERS, LAYERS]  # 用受控断言验证关键不变量。
assert ((0 <= late_token) & (late_token < VOCAB)).all()  # 用受控断言验证关键不变量。


## 6. Self-speculative：浅层提案，完整层验证最长一致前缀

浅层一次提出多个 token，完整模型批量验证；greedy 情况只提交与 full argmax 连续一致的前缀，首个分歧后停止。随机采样则要接受—拒绝/残差分布校正，不能照搬 argmax。


In [ ]:
def commit_verified_prefix(draft_tokens, verified_tokens):  # 定义本节可复用的核心函数。
    if len(draft_tokens) != len(verified_tokens):  # 按当前条件选择后续控制路径。
        raise ValueError("draft 与 verifier block 长度必须相同")  # 遇到非法合同立即显式失败。
    accepted = 0  # 计算并保存当前步骤的中间状态。
    while accepted < len(draft_tokens) and draft_tokens[accepted] == verified_tokens[accepted]:  # 在终止条件满足前持续推进状态。
        accepted += 1  # 计算并保存当前步骤的中间状态。
    committed = list(draft_tokens[:accepted])  # 计算并保存当前步骤的中间状态。
    if accepted < len(draft_tokens):  # 按当前条件选择后续控制路径。
        committed.append(verified_tokens[accepted])  # 首个 mismatch 提交 verifier token，保证前进。
    return accepted, committed  # 返回当前分支计算出的结果。

@torch.no_grad()  # 为下方定义附加声明式配置。
def self_speculative_verify(model, token_ids, valid_mask, positions, draft_depth):  # 定义本节可复用的核心函数。
    if token_ids.shape[0] != 1 or not 0 < draft_depth < len(model.layers):  # 按当前条件选择后续控制路径。
        raise ValueError("教学 verifier 要求 batch=1 且 draft_depth 位于中间层")  # 遇到非法合同立即显式失败。
    hidden = model.prefix_hidden(token_ids, valid_mask)  # 计算并保存当前步骤的中间状态。
    shallow_hidden, _ = model.run_layers(hidden, start=0, stop=draft_depth)  # 计算并保存当前步骤的中间状态。
    draft = model.logits_from(shallow_hidden)[0, positions].argmax(-1).tolist()  # 计算并保存当前步骤的中间状态。
    verified_hidden, _ = model.run_layers(shallow_hidden, start=draft_depth, stop=len(model.layers))  # 计算并保存当前步骤的中间状态。
    verified = model.logits_from(verified_hidden)[0, positions].argmax(-1).tolist()  # 计算并保存当前步骤的中间状态。
    accepted, committed = commit_verified_prefix(draft, verified)  # 计算并保存当前步骤的中间状态。
    return {"draft": draft, "verified": verified, "accepted": accepted, "committed": committed}  # 返回当前分支计算出的结果。

# 主路径先跑浅层 draft，再复用该 hidden 只跑剩余层；结果必须等于完整模型最终出口。
positions = torch.tensor([1, 2, 3])  # 计算并保存当前步骤的中间状态。
speculation = self_speculative_verify(model, tokens[:1], valid[:1], positions, draft_depth=2)  # 计算并保存当前步骤的中间状态。
full_tokens = exit_logits[-1][0, positions].argmax(-1).tolist()  # 计算并保存当前步骤的中间状态。
assert speculation["verified"] == full_tokens  # 用受控断言验证关键不变量。
assert speculation["accepted"] < len(positions)  # 固定 seed 下真实浅/深层在 block 内发生 mismatch。
assert speculation["committed"][-1] == speculation["verified"][speculation["accepted"]]  # 用受控断言验证关键不变量。

# 明确反例：第二个 token mismatch 时，只接收首 token，并提交 verifier 的 mismatch token 4。
accepted_probe, committed_probe = commit_verified_prefix([3, 9, 5], [3, 4, 5])  # 计算并保存当前步骤的中间状态。
assert accepted_probe == 1  # 用受控断言验证关键不变量。
assert committed_probe == [3, 4]  # 用受控断言验证关键不变量。
assert commit_verified_prefix([3, 4, 5], [3, 4, 5]) == (3, [3, 4, 5])  # 用受控断言验证关键不变量。


## 7. 计算模型：接受率、exit depth 与共享计算共同决定收益

若 draft 用 e 层、full 用 L 层，拒绝过多会重复剩余计算；共享早期 hidden/KV 可降低开销。理论层数比例不包含 kernel、batch、验证宽度、内存带宽和同步。


In [ ]:
def estimated_layers_per_committed_token(total_layers, exit_layer, block_size, accepted):  # 定义本节可复用的核心函数。
    if not 0 <= accepted <= block_size:  # 按当前条件选择后续控制路径。
        raise ValueError("accepted 必须位于 block 范围")  # 遇到非法合同立即显式失败。
    draft_cost = exit_layer * block_size  # 计算并保存当前步骤的中间状态。
    verify_cost = (total_layers - exit_layer) * block_size  # 计算并保存当前步骤的中间状态。
    # 有 mismatch 时额外提交一个 verifier token；全部接受时提交整个 draft block。
    committed = accepted + int(accepted < block_size)  # 计算并保存当前步骤的中间状态。
    return (draft_cost + verify_cost) / max(1, committed)  # 返回当前分支计算出的结果。

# 接受更多 token 会降低每提交 token 的等效层数；零接受与全接受都不会除零。
cost0 = estimated_layers_per_committed_token(LAYERS, 2, 4, 0)  # 计算并保存当前步骤的中间状态。
cost3 = estimated_layers_per_committed_token(LAYERS, 2, 4, 3)  # 计算并保存当前步骤的中间状态。
assert cost3 < cost0  # 用受控断言验证关键不变量。
assert cost0 == LAYERS * 4  # 用受控断言验证关键不变量。
assert estimated_layers_per_committed_token(LAYERS, 2, 4, 4) == LAYERS  # 用受控断言验证关键不变量。


## 8. 发布门禁：训练出口与推理解码器必须同版本

manifest 绑定 layer-drop schedule、exit-loss 权重、共享 head、draft depth、block size、采样校正规则和 KV layout。验收报告最终质量、各层校准、接受长度、TTFT/TPOT、显存与失败回退。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class LayerSkipArtifact:  # 定义承载本节状态与行为的数据结构。
    layers: int  # 执行当前语句以推进本节示例。
    exit_loss: str  # 执行当前语句以推进本节示例。
    dropout_schedule: str  # 执行当前语句以推进本节示例。
    draft_depth: int  # 执行当前语句以推进本节示例。
    verifier: str  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# draft depth 必须小于总层数，任何 verifier 变化产生新摘要。
artifact = LayerSkipArtifact(LAYERS, "weighted-all-exits-v1", "executed-linear-drop-v2", 2, "reuse-hidden-greedy-commit-v2")  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert 0 < artifact.draft_depth < artifact.layers  # 用受控断言验证关键不变量。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert digest != artifact_hash(LayerSkipArtifact(LAYERS, artifact.exit_loss, artifact.dropout_schedule, 2, "sampling-residual-v3"))  # 用受控断言验证关键不变量。


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
